# IDSW v2.0.0 Workflow: Recommendation Engines (`idsw.modelling.recommendation_engines`)

This notebook demonstrates the new recommendation engines submodule in **IDSW 2.0.0**:
- Transforming wide user-item matrices into long rating tables (`wide_to_long_ratings`).
- Alternating Least Squares (ALS) matrix factorization (`als_recommender`) supporting both PySpark MLlib and pure-NumPy fallback for explicit and implicit feedback.
- Generating top-N recommendations for users (`recommend_for_users`) and items (`recommend_for_items`).
- User-based and item-based collaborative filtering (`collaborative_filtering_recommender`).

See `als_recommender.__doc__` for step-by-step instructions on configuring PySpark and SparkMagic on Windows 11.

## Setup Guide: Installing PySpark and SparkMagic Kernel on Windows 11

1. **Install Java**: Download OpenJDK 11 or 17 (e.g., Eclipse Temurin) and set JAVA_HOME, adding %JAVA_HOME%\bin to PATH.
2. **Install Apache Spark**: Download Spark 3.5+ (pre-built for Hadoop 3.3+), extract to C:\spark, and set SPARK_HOME=C:\spark.
3. **Windows Hadoop Binaries (winutils.exe)**: Place winutils.exe and hadoop.dll in C:\hadoop\bin and set HADOOP_HOME=C:\hadoop.
4. **Install Python Packages**: pip install pyspark findspark sparkmagic
5. **Register SparkMagic Kernel**: jupyter nbextension enable --py --sys-prefix widgetsnbextension, then jupyter-kernelspec install --user [sparkmagic_dir]/kernels/pysparkkernel


In [ ]:
import numpy as np
import pandas as pd
from idsw import *

ControlVars.show_results = True

In [ ]:
# Create a sample interaction dataset (wide format -> long format)
wide_df = pd.DataFrame({
    'user_id': [1, 2, 3, 4],
    'item_101': [5.0, 4.0, np.nan, 1.0],
    'item_102': [3.0, np.nan, 5.0, 2.0],
    'item_103': [np.nan, 5.0, 4.0, 5.0],
    'item_104': [1.0, 2.0, np.nan, 4.0]
})

ratings_df = wide_to_long_ratings(
    df=wide_df,
    user_col='user_id',
    item_col_name='item_id',
    rating_col_name='rating'
)

# Train ALS Recommender (uses PySpark if available, or pure NumPy ALS fallback)
als_model, als_metrics, als_preds = als_recommender(
    df=ratings_df,
    user_col='user_id',
    item_col='item_id',
    rating_col='rating',
    rank=5,
    max_iter=10,
    reg_param=0.1,
    implicit_prefs=False,
    use_pyspark=True
)

# Top-N recommendations for users and items
user_recs = recommend_for_users(als_model, n_recommendations=3)
item_recs = recommend_for_items(als_model, n_recommendations=3)

In [ ]:
# Collaborative Filtering Recommender (Item-based / User-based Cosine Similarity)
cf_model, cf_preds = collaborative_filtering_recommender(
    df=ratings_df,
    user_col='user_id',
    item_col='item_id',
    rating_col='rating',
    method='item_based',
    k_neighbors=2
)